<a href="https://colab.research.google.com/github/frandresst-wq/Proyecto-de-Ciencia-de-Datos/blob/main/S1_Notebook_Base_CDD_Soto_Franco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 1 — Obtención del dato
**Ciencia de Datos · Ingeniería de Ejecución en Informática — Advance UCSC**

| | |
|---|---|
| **Semana** | 1 |
| **Resultado de aprendizaje** | RA1 (foco) · RA3 (transversal) · RA5 (apertura) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Estudiante** | _(escribe tu nombre aquí)_ |

---

## Antes de empezar

Sube al entorno los tres archivos que acompañan esta guía:

- `S1_Dataset_Guia_CDD.csv`
- `S1_Base_Ejemplo_CDD.sqlite`
- `S1_Respuesta_API_Ejemplo_CDD.json`

En Google Colab: panel izquierdo → icono de carpeta → *Subir*. En Jupyter local: déjalos en la misma carpeta que este cuaderno.

## Celda 0 · Pregunta de activación

Antes de escribir código, responde aquí para ti (no se entrega):

> **¿Qué dato de tu propio trabajo no usarías hoy para tomar una decisión importante, y por qué?**

_Escribe tu respuesta reemplazando esta línea._

## 1 · Preparación del entorno

Un flujo reproducible parte por declarar qué versiones se usaron. Si alguien ejecuta este cuaderno dentro de un año y obtiene otro resultado, lo primero que va a mirar es esto.

In [ ]:
import sys, platform
import pandas as pd
import numpy as np

RUTA = ''          # en Colab déjalo vacío; en local, la carpeta donde están los archivos
SEMILLA = 20261
np.random.seed(SEMILLA)

print('Python    :', sys.version.split()[0], '·', platform.system())
print('pandas    :', pd.__version__)
print('numpy     :', np.__version__)
print('semilla   :', SEMILLA)

## 2 · Fuente 1 — Archivo plano (CSV)

El primer problema de un archivo plano nunca es el análisis: es **abrirlo bien**. Tres cosas pueden salir mal antes de ver un solo número: la codificación, el separador y los tipos de dato.

Ejecuta la celda siguiente y observa el resultado. Después responde: ¿cuál es el separador de este archivo?

In [ ]:
# Lectura deliberadamente ingenua: mira lo que pasa
try:
    crudo = pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv')
    print('Forma:', crudo.shape)
    display(crudo.head(3))
except Exception as e:
    print(type(e).__name__)
    print(e)

**Lee el error, no lo saltes.** Dice algo como *«Expected 1 fields in line 39, saw 2»*.

Se traduce así: pandas asumió que el separador era la coma, no encontró ninguna en la primera línea y concluyó que el archivo tenía **una sola columna**. Todo iba bien hasta la línea 39, donde apareció una coma —un decimal escrito a la chilena, `12,4`— y esa fila pasó a tener dos campos. Contradicción, y se detiene.

Son dos defectos a la vez: el separador real es `;` y además hay decimales con coma. Resolvamos primero el separador.

In [ ]:
aire = pd.read_csv(
    RUTA + 'S1_Dataset_Guia_CDD.csv',
    sep=';',
    encoding='utf-8',
)
print('Forma:', aire.shape)
aire.head()

In [ ]:
# Tipos de dato con los que pandas interpretó cada columna
aire.dtypes

### ✏️ Actividad 2.1

Mira los tipos con atención. `pm25` y `pm10` quedaron numéricos, pero `temperatura_c` quedó como `object`, es decir, texto. Que una columna numérica quede como texto **siempre** significa que hay algo dentro que no es un número: aquí son los nueve decimales escritos con coma que provocaron el error anterior.

Y existe la trampa inversa: `pm10` es numérico y aun así está contaminado. Lo verás en la actividad 3.2.

**Escribe el código** que muestre los valores no numéricos de `temperatura_c`. Pista: `pd.to_numeric(..., errors='coerce')` convierte a `NaN` lo que no puede convertir.

In [ ]:
# Tu código aquí


## 3 · Perfilado inicial: ¿qué tengo entre manos?

Antes de analizar, hay que **conocer el dato**. Estas cuatro celdas son el mínimo que un analista ejecuta siempre, en este orden.

In [ ]:
# a) Dimensiones y nombres de columna
print('Filas   :', aire.shape[0])
print('Columnas:', aire.shape[1])
print()
print(aire.columns.tolist())

In [ ]:
# b) Valores faltantes por columna
faltantes = aire.isna().sum().to_frame('faltantes')
faltantes['porcentaje'] = (faltantes['faltantes'] / len(aire) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

In [ ]:
# c) Filas duplicadas completas
print('Duplicados exactos:', aire.duplicated().sum())
print('id_registro repetidos:', aire['id_registro'].duplicated().sum())

In [ ]:
# d) Cardinalidad de las columnas categóricas
for col in ['estacion', 'comuna', 'region', 'validado']:
    print(f'{col:15s} · {aire[col].nunique():3d} valores distintos')

### ✏️ Actividad 3.1

La columna `comuna` declara más valores distintos de los que deberían existir. **Ejecuta** `aire['comuna'].unique()` y explica en la celda de texto siguiente por qué ocurre.

Ojo: no lo corrijas todavía. Esta semana **detectamos**; en la semana 2 **corregimos** y dejamos registrada la decisión en la bitácora.

In [ ]:
aire['comuna'].unique()

_Tu explicación aquí:_



### ✏️ Actividad 3.2 — Valores centinela

Un **valor centinela** es un número real que en realidad significa «no hay dato»: `-999`, `0`, `9999`. Si no lo detectas, entra al promedio y lo arruina.

**Escribe el código** que cuente cuántas veces aparece `-999` en `pm10` y calcula el promedio de `pm10` con y sin esos valores. Anota la diferencia.

In [ ]:
# Tu código aquí


## 4 · Fuente 2 — Base de datos relacional (SQL)

Buena parte de los datos de una organización no está en archivos, sino en una base de datos. El archivo `S1_Base_Ejemplo_CDD.sqlite` contiene tres tablas: `estacion`, `medicion` y `episodio`.

SQLite viene incluido en Python: no hay que instalar nada.

In [ ]:
import sqlite3

con = sqlite3.connect(RUTA + 'S1_Base_Ejemplo_CDD.sqlite')

# ¿Qué tablas hay?
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", con)

In [ ]:
# Estructura de una tabla
pd.read_sql_query('PRAGMA table_info(medicion)', con)

In [ ]:
# Una consulta con JOIN y agregación: promedio de PM2.5 por comuna
consulta = '''
SELECT  e.comuna,
        e.region,
        COUNT(*)              AS n_mediciones,
        ROUND(AVG(m.pm25), 1) AS pm25_promedio,
        ROUND(MAX(m.pm25), 1) AS pm25_maximo
FROM    medicion m
JOIN    estacion e ON e.estacion_id = m.estacion_id
GROUP BY e.comuna, e.region
ORDER BY pm25_promedio DESC
'''
pd.read_sql_query(consulta, con)

### ✏️ Actividad 4.1

**Escribe una consulta SQL** que responda: *¿cuántos episodios de cada categoría (`Alerta`, `Preemergencia`, `Emergencia`) registró cada comuna?*

Necesitas unir `episodio` con `estacion` y agrupar por dos columnas.

In [ ]:
consulta_2 = '''
-- Tu consulta aquí
'''
# pd.read_sql_query(consulta_2, con)

In [ ]:
con.close()

## 5 · Fuente 3 — Servicio web (API) y JSON anidado

Una API devuelve **JSON**, no una tabla. El JSON es jerárquico: dentro de un registro puede haber otro objeto, y dentro de ese, una lista. Convertirlo en una tabla rectangular es parte del trabajo.

Usamos aquí la respuesta guardada en archivo. En un caso real la obtendrías con `requests.get(url).json()`; el archivo existe para que el laboratorio funcione aunque el servicio esté caído.

In [ ]:
import json

with open(RUTA + 'S1_Respuesta_API_Ejemplo_CDD.json', encoding='utf-8') as fh:
    respuesta = json.load(fh)

print('Claves de primer nivel:', list(respuesta.keys()))
print()
print('Metadatos:')
for k, v in respuesta['meta'].items():
    print(f'  {k}: {v}')

In [ ]:
# Un registro cualquiera, para entender la estructura
print(json.dumps(respuesta['data'][0], ensure_ascii=False, indent=2))

In [ ]:
# json_normalize aplana la jerarquía: fíjate en los nombres de columna que genera
api = pd.json_normalize(respuesta['data'])
api.head(3)

In [ ]:
# La lista 'mediciones' sigue anidada. record_path la despliega en filas:
api_largo = pd.json_normalize(
    respuesta['data'],
    record_path='mediciones',
    meta=['codigo_estacion', 'fecha', ['ubicacion', 'comuna']],
)
api_largo.head(6)

### ✏️ Actividad 5.1

El resultado anterior está en **formato largo**: una fila por parámetro medido. Para analizar conviene el **formato ancho**: una fila por estación-fecha y una columna por parámetro.

**Escribe el código** que haga esa transformación con `pivot_table`.

In [ ]:
# Tu código aquí


## 6 · Fuente 4 — Extracción web

Cuando el dato solo existe publicado en una página, hay que extraerlo. `pandas.read_html` resuelve el caso más común: una tabla HTML.

**Antes de extraer, tres preguntas que no son técnicas:** ¿los términos de uso del sitio lo permiten?, ¿estás generando una carga razonable?, ¿el dato incluye personas? Si alguna respuesta te incomoda, no extraigas.

In [ ]:
# Requiere conexión a internet. Si no la tienes, salta esta celda.
URL = 'https://es.wikipedia.org/wiki/Anexo:Comunas_de_Chile'   # [DOCENTE: confirmar o reemplazar]

try:
    tablas = pd.read_html(URL)
    print('Tablas encontradas:', len(tablas))
    web = tablas[0]
    display(web.head())
except Exception as e:
    print('No se pudo extraer:', type(e).__name__, e)
    print('Continúa con el resto del laboratorio.')

### ✏️ Actividad 6.1

Responde en la celda de texto siguiente: **¿qué diferencia hay, para tu proyecto, entre un dato descargado de un portal oficial y uno extraído de una página web?** Piensa en trazabilidad y en qué pasa si mañana la página cambia.

_Tu respuesta aquí:_



## 7 · Volumen: cuando el conjunto no cabe

No todo conjunto se lee entero. Cuando el archivo supera la memoria disponible hay tres salidas: **leer solo las columnas necesarias**, **leer por bloques** o **trabajar con una muestra**.

Una muestra bien construida responde muchas preguntas con una fracción del costo. Una muestra mal construida responde mal todas.

In [ ]:
# Leer solo algunas columnas: menos memoria desde el inicio
liviano = pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv', sep=';',
                      usecols=['estacion', 'fecha', 'pm25'])
print('Memoria completo :', aire.memory_usage(deep=True).sum() / 1024, 'KB')
print('Memoria liviano  :', liviano.memory_usage(deep=True).sum() / 1024, 'KB')

In [ ]:
# Lectura por bloques: se procesa de a trozos y nunca se carga todo
total, filas = 0, 0
for bloque in pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv', sep=';', chunksize=100):
    valores = pd.to_numeric(bloque['pm25'], errors='coerce').dropna()
    total += valores.sum()
    filas += len(valores)
print(f'Promedio de pm25 calculado por bloques: {total / filas:.2f}')

In [ ]:
# Muestreo aleatorio simple vs. estratificado por estación
simple = aire.sample(n=120, random_state=SEMILLA)
estratificado = (aire.groupby('estacion', group_keys=False)
                     .apply(lambda g: g.sample(n=15, random_state=SEMILLA)))

print('Distribución en la muestra SIMPLE:')
print(simple['estacion'].value_counts().to_string())
print()
print('Distribución en la muestra ESTRATIFICADA:')
print(estratificado['estacion'].value_counts().to_string())

### ✏️ Actividad 7.1

Mira las dos distribuciones anteriores. **¿En qué situación la muestra simple te llevaría a una conclusión equivocada?** Escríbelo abajo con un ejemplo concreto de tu propio conjunto de datos.

_Tu respuesta aquí:_



## 8 · Tu conjunto de datos: la elección del proyecto

Abre `S1_Catalogo_Datasets_Sugeridos_CDD.xlsx`, revisa **al menos tres** alternativas y elige una. Registra tu elección en la hoja «Mi elección» del mismo archivo.

Luego descárgalo, cárgalo aquí abajo y ejecuta sobre él el mismo perfilado de la sección 3.

In [ ]:
# mi_dataset = pd.read_csv(RUTA + 'NOMBRE_DE_TU_ARCHIVO.csv')
# print(mi_dataset.shape)
# mi_dataset.head()


In [ ]:
# Perfilado de tu conjunto: repite aquí las cuatro celdas de la sección 3


### ✏️ Actividad 8.1 — Pregunta analítica y unidad de análisis

Completa estas dos líneas. Son la base de todo el curso y las vas a necesitar en la Entrega 1.

- **Mi pregunta analítica es:** _…_
- **Una fila de mi conjunto representa:** _…_

Criterio para saber si la pregunta está bien formulada: si al leerla puedes nombrar qué columnas necesitas para responderla, está bien. Si no, es demasiado vaga.

## 9 · Riesgo ético y legal

Antes de usar tu conjunto, completa la ficha `S1_Ficha_Riesgo_Etico_CDD.docx`. No es un trámite: si tu conjunto contiene datos de personas, lo que decidas ahí condiciona todo lo que puedes hacer en las semanas siguientes.

Esa ficha es el insumo de tu intervención en el **foro de debate** de esta semana y se cierra en la Sección C de la Entrega Final.

## 10 · Repositorio y control de versiones

Tu trabajo tiene que poder reconstruirse. Sube este cuaderno a un repositorio y deja el enlace aquí.

Desde Colab: *Archivo → Guardar una copia en GitHub*. Desde local:

In [ ]:
# Ejecuta esto en una terminal, no en el cuaderno:
# git init
# git add S1_Notebook_Base_CDD.ipynb
# git commit -m 'Laboratorio 1: obtención del dato'
# git remote add origin https://github.com/USUARIO/proyecto-ciencia-datos.git
# git push -u origin main


**Enlace a mi repositorio:** _…_

---

## Cierre del laboratorio

Al terminar deberías tener:

- [ ] El entorno funcionando y las versiones declaradas
- [ ] Datos leídos desde las cuatro fuentes
- [ ] El perfilado inicial del conjunto guía, con sus defectos detectados (no corregidos)
- [ ] Tu conjunto de datos elegido y registrado en el catálogo
- [ ] Tu pregunta analítica y tu unidad de análisis escritas
- [ ] La ficha de riesgo ético-legal completada
- [ ] El repositorio creado y este cuaderno versionado

Lo que quede pendiente, llévalo a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 1_